# Solicitação de Análise de Especialistas

Nome: Fabrício Camacho

O grupo de especialistas da FGV DGPE solicitou uma análise dos resultados atuais do PBA. A presente análise tem como objetivo:
- Analisar se faz sentido estudar as questões de cada Prática de Linguagem separadamente ao invés de dar a nota atribuida somente a prática.
- Entender se há uma relação entre estudantes que se mantém em Nível 1 com a idade;
- Entender se a demora na mudança de nível está relacionada a idade ou não.

Após importar os dados do sistema, será necessário tratar os dados e extrair somente as informações de turmas concluídas, para assim podermos trabalhar com dados completos. Além disso, os dados do PBA SE e RN serão marcados e unidos para que tenhamos a possibilidade de trabalhar analisando os dados juntos e em separado e verificar se temos variações significativas.

In [1]:
# Importando bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Importando arquivo
df_alfabetizandos_se = pd.read_excel(
    'Data_files/public_data/xls_alfabetizandos_se_27102025.xlsx')
df_alfabetizandos_rn = pd.read_excel(
    'Data_files/public_data/xls_alfabetizandos_rn_27102025.xlsx')
df_cadastro_se = pd.read_excel(
    'Data_files/public_data/xls_se_pedagogico_27102025.xlsx')
df_cadastro_rn = pd.read_excel(
    'Data_files/public_data/xls_rn_pedagogico_27102025.xlsx')
df_turmas_se = pd.read_excel(
    'Data_files/public_data/se_turmas_27102025.xlsx')
df_turmas_rn = pd.read_excel(
    'Data_files/public_data/rn_turmas_27102025.xlsx')
df_registros_se = pd.read_excel(
    'Data_files/public_data/xls_se_detalhado_27102025.xlsx')
df_registros_rn = pd.read_excel(
    'Data_files/public_data/xls_rn_detalhado_27102025.xlsx')

# Imprimindo informações básicas dos dataframes
print(df_alfabetizandos_se.info())
print()
print(df_alfabetizandos_rn.info())
print()
print(df_cadastro_se.info())
print()
print(df_cadastro_rn.info())
print()
print(df_turmas_se.info())
print()
print(df_turmas_rn.info())
print()
print(df_registros_se.info())
print()
print(df_registros_rn.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11271 entries, 0 to 11270
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   CPF                 11271 non-null  object
 1   Nome                11271 non-null  object
 2   Telefone            11039 non-null  object
 3   Data de nascimento  8892 non-null   object
 4   DRE                 11244 non-null  object
 5   Município           11244 non-null  object
 6   Bairro              9693 non-null   object
 7   Endereço completo   9425 non-null   object
 8   Turno               11271 non-null  object
 9   E-mail              7975 non-null   object
 10  Turma               11271 non-null  object
 11  Data de inserção    11271 non-null  object
 12  CPF validado        8963 non-null   object
dtypes: object(13)
memory usage: 1.1+ MB
None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1885 entries, 0 to 1884
Data columns (total 33 columns):
 #   Column      

## Tratando dos dados
Se faz necessário a transformação dos dados para que seja mais fácil sua análise. De acordo com a observação dos dados apresentados, as seguintes transformações serão necessárias:
Em df_registros:
- Transformar nomes de cada coluna para o padrão snake_case;
- Extrair somente os níveis da coluna "Nível";
- Deixar os níveis de maneira categórica corretamente para melhor análise na coluna "Nível".

Em df_cadastro, df_turmas:
- Todas as colunas que contem datas deverão ser tranformados para o tipo datetime.

Além disso, vamos criar uma coluna a mais em cada dataframe para sinalizar o estado em que os dados fazem parte e criar dfs únicos para análise geral, sem perder a informação de que estado eles são oriundos.

In [2]:
# Transformando nomes de colunas para padrão snake_case
# Função para converter para snake_case
def to_snake_case(column_name):
    return (
        column_name
        .lower()  # Converte tudo para minúsculo
        .replace(' ', '_')  # Substitui espaços por underscore
        .replace('é', 'e')  # Remove acentos
        .replace('ã', 'a')
        .replace('ç', 'c')
        .replace('í', 'i')
        .replace('ó', 'o')
        .replace('ú', 'u')
        .replace('á', 'a')
        .replace('.', '')  # Remove pontos
        .replace('-', '')  # Substitui hífens por underscore
        .replace('(', '')  # Remove parênteses
        .replace(')', '')  # Remove parênteses
        .replace('__', '_')  # Substitui duplo underscore por simples
    )

# Função para extrair somente o nível de df_registros e classifica-los categoricamente
def extract_level(df):
    df['nivel'] = df['nivel'].str.extract(r'(Nível [1-3])')
    df['nivel'] = pd.Categorical(
        df['nivel'],
        categories=['Nível 1', 'Nível 2', 'Nível 3'], ordered=True)
    return df

# Função para transformar dados em datetime
def convert_to_datetime(df, column_name):
    df[column_name] = pd.to_datetime(df[column_name], dayfirst=True, errors='coerce')
    return df

# Função para marcar os dados do PBA SE e RN
def mark_project(df, project_code):
    df['projeto'] = project_code
    return df

# Aplicando as transformações
dataframes = {
    'df_alfabetizandos_se': df_alfabetizandos_se,
    'df_alfabetizandos_rn': df_alfabetizandos_rn,
    'df_cadastro_se': df_cadastro_se,
    'df_cadastro_rn': df_cadastro_rn,
    'df_turmas_se': df_turmas_se,
    'df_turmas_rn': df_turmas_rn,
    'df_registros_se': df_registros_se,
    'df_registros_rn': df_registros_rn
}

for name, df in dataframes.items():
    # Renomeando colunas para snake_case
    df.columns = [to_snake_case(col) for col in df.columns]
    
    # Extraindo nível e classificando categoricamente para df_registros
    if 'registros' in name:
        df = extract_level(df)
    
    # Convertendo colunas de data para datetime
    date_columns = [col for col in df.columns if 'data_' in col or 'data ' in col]
    for date_col in date_columns:
        df = convert_to_datetime(df, date_col)
        
    # Marcando os dados do PBA SE e RN
    if 'se' in name:
        df = mark_project(df, 'PBA_SE')
    elif 'rn' in name:
        df = mark_project(df, 'PBA_RN')
    
    # Atualizando o dataframe no dicionário
    dataframes[name] = df


Agora vamos filtrar os dataframes somente com dados de turmas concluídas até o momento da coleta de dados - 27/10/2025

In [3]:
# Mantendo dados de turmas concluídas
# Listando turmas concluídas
turmas_rn = df_turmas_rn[df_turmas_rn['situacao_da_turma'] == 'Concluída']['turma'].to_list()
turmas_se = df_turmas_se[df_turmas_se['situacao_da_turma'] == 'Concluída']['turma'].to_list()

# Filtrando dataframes para manter somente turmas concluídas
df_alfabetizandos_rn = df_alfabetizandos_rn[df_alfabetizandos_rn['turma_formacao'].isin(turmas_rn)] #coluna com nome diferente
df_alfabetizandos_se = df_alfabetizandos_se[df_alfabetizandos_se['turma'].isin(turmas_se)]
df_cadastro_rn = df_cadastro_rn[df_cadastro_rn['turma'].isin(turmas_rn)]
df_cadastro_se = df_cadastro_se[df_cadastro_se['turma'].isin(turmas_se)]
df_turmas_rn = df_turmas_rn[df_turmas_rn['turma'].isin(turmas_rn)]
df_turmas_se = df_turmas_se[df_turmas_se['turma'].isin(turmas_se)]
df_registros_rn = df_registros_rn[df_registros_rn['turma'].isin(turmas_rn)]
df_registros_se = df_registros_se[df_registros_se['turma'].isin(turmas_se)]

# Ajustando df_alfabetizandos_rn para manter somente colunas relevantes
df_alfabetizandos_rn = df_alfabetizandos_rn.drop(columns=df_alfabetizandos_rn.columns[[10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]])
df_alfabetizandos_rn = df_alfabetizandos_rn.rename(columns={'turma_formacao': 'turma'})

#Concatenando dataframes do PBA SE e RN
df_alfabetizandos_geral = pd.concat([df_alfabetizandos_se, df_alfabetizandos_rn], ignore_index=True)
df_cadastro_geral = pd.concat([df_cadastro_se, df_cadastro_rn], ignore_index=True)
df_turmas_geral = pd.concat([df_turmas_se, df_turmas_rn], ignore_index=True)
df_registros_geral = pd.concat([df_registros_se, df_registros_rn], ignore_index=True)

# Imprimindo resultados
print(df_alfabetizandos_geral.info())
print()
print(df_cadastro_geral.info())
print()
print(df_turmas_geral.info())
print()
print(df_registros_geral.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4417 entries, 0 to 4416
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   cpf                 4417 non-null   object        
 1   nome                4417 non-null   object        
 2   telefone            4217 non-null   object        
 3   data_de_nascimento  4341 non-null   datetime64[ns]
 4   dre                 4260 non-null   object        
 5   municipio           4411 non-null   object        
 6   bairro              4233 non-null   object        
 7   endereco_completo   4217 non-null   object        
 8   turno               4417 non-null   object        
 9   email               3833 non-null   object        
 10  turma               4417 non-null   object        
 11  data_de_insercao    4417 non-null   datetime64[ns]
 12  cpf_validado        4410 non-null   object        
 13  projeto             4417 non-null   object      

### Gerando novas informações
Aqui vamos calcular a idade dos alfabetizandos com base na data de nascimento informada. Faremos as seguintes etapas:
1. Extrair o ano daqueles que informaram data de nascimento;
2. Diminuir o ano atual (2025) do ano de nascimento para entender a idade atual do alfabetizando;
3. Desconsiderar idades menores que 15 anos por considerar data errada, uma vez que o programa é destinado para pessoas com 15 anos ou mais;
4. Criar categorias de idade de 10 em 10 anos da seguinte forma:
    a. 15 a 24 anos;
    b. 25 a 34 anos;
    c. 35 a 44 anos;
    d. 45 a 54 anos;
    e. 55 a 64 anos;
    f. 65 a 74 anos;
    g. 75 anos ou mais.

In [4]:
# Excluindo dados de alfabetizandos que não possuem data de nascimento
alfabetizandos_sem_dt_nascimento = df_alfabetizandos_geral[df_alfabetizandos_geral['data_de_nascimento'].isnull()]
df_alfabetizandos_geral = df_alfabetizandos_geral.dropna(subset=['data_de_nascimento'])

# Extraindo o ano de nascimento dos alfabetizandos
df_alfabetizandos_geral['ano_nascimento'] = df_alfabetizandos_geral['data_de_nascimento'].dt.year

# Calculando a idade dos alfabetizandos considerando o ano de 2025
df_alfabetizandos_geral['idade'] = 2025 - df_alfabetizandos_geral['ano_nascimento']

# Excluindo dados de alunos menores de 15 anos
alfabetizandos_menores_de_15 = df_alfabetizandos_geral[df_alfabetizandos_geral['idade'] < 15]
df_alfabetizandos_geral = df_alfabetizandos_geral[df_alfabetizandos_geral['idade'] > 14]

# Verificando tamanho da amostra
print(f'Alfabetizandos sem data de nascimento: {len(alfabetizandos_sem_dt_nascimento)}')
print(f'Alfabetizandos menores de 15 anos: {len(alfabetizandos_menores_de_15)}')
print(f'Alfabetizando com dados completos: {len(df_alfabetizandos_geral)}')


Alfabetizandos sem data de nascimento: 76
Alfabetizandos menores de 15 anos: 23
Alfabetizando com dados completos: 4318


Um total de 98 dados foram retirados devido a ausência de data de nascimento (85) ou por se tratar de um estudante menor de 15 anos (13). Dessa forma, mantivemos um total de 4319 estudantes na base, o que será o suficiente para a análise que faremos.
Agora, vamos criar categorias de idade e marcar os estudantes nas devidas categorias.

In [5]:
# Criando categoria de idade para cada estudante
# Criando cortes (bins) para cada categoria de idade
bins = [
    14,             # Borda 0 (0 a 14 anos)
    24,             # Borda 1 (15 a 24 anos)
    34,             # Borda 2 (25 a 34 anos)
    44,             # Borda 3 (35 a 44 anos)
    54,             # Borda 4 (45 a 54 anos)
    64,             # Borda 5 (55 a 64 anos)
    74,             # Borda 6 (65 a 74 anos)
    float('inf')    # Borda 7 (mais de 74 anos)
]

# Definindo categorias para cada faixa de idade
age_categories = [
    '15 a 24 anos',
    '25 a 34 anos',
    '35 a 44 anos',
    '45 a 54 anos',
    '55 a 64 anos',
    '65 a 74 anos',
    '75 anos ou mais'
]

# Criando nova coluna com categorias de idade
df_alfabetizandos_geral['idade_categoria'] = pd.cut(
    df_alfabetizandos_geral['idade'],
    bins=bins,
    labels=age_categories)

# Calculando o número de alunos por faixa de idade
print(df_alfabetizandos_geral.groupby('idade_categoria', observed=True)['cpf'].count())


idade_categoria
15 a 24 anos       598
25 a 34 anos       549
35 a 44 anos       600
45 a 54 anos       567
55 a 64 anos       574
65 a 74 anos       552
75 anos ou mais    878
Name: cpf, dtype: int64
